## Part 8 – Frequency Analysis Adjustment

In this section we take a look at how far off the simple letter-frequency method actually is. 
The idea is to compare:

- the decode we get from the basic frequency mapping, and  
- the decode we get after running the Metropolis/bigram method.

By comparing the two, we can get a sense of how many cipher letters the simple method gets wrong, 
and how much adjustment the Metropolis method has to make.


In [ ]:
# Part 8 – checking how far off simple frequency analysis actually is
# The idea here is:
#   - get a decode from the basic letter-frequency method
#   - get a decode from the Metropolis/bigram method
#   - compare the two to see how many cipher letters change their mapping
#   - also measure how different the two decodes are position-by-position
#
# This gives us a rough idea of "how much" the simple approach needs to be fixed.

from collections import Counter

# quick helper: infer a mapping from (ciphertext, decoded_text)
def get_mapping_from_decoded(ciphertext, decoded_text):
    m = {}
    for c in character_list:
        targets = []
        for i, ch in enumerate(ciphertext):
            if ch == c and i < len(decoded_text):
                targets.append(decoded_text[i])
        if targets:
            # majority vote, nothing fancy
            m[c] = Counter(targets).most_common(1)[0][0]
        else:
            m[c] = c   # if not present in message, leave it mapped to itself
    return m

# compare how many letters map to the same plaintext
def compare_maps(m1, m2):
    same, diff = 0, 0
    for c in character_list:
        if m1[c] == m2[c]:
            same += 1
        else:
            diff += 1
    return same, diff

# check how many positions match in the two decoded strings
def positional_overlap(t1, t2):
    n = min(len(t1), len(t2))
    if n == 0:
        return 0.0
    matches = sum(1 for i in range(n) if t1[i] == t2[i])
    return matches / n


# ------- run the comparison on one ciphertext --------
# (can change this to any coded_message_* we want)
ciphertext = coded_message_02

# simple frequency-based decode
english_freqs = get_frequency(moby_new)
freq_map = get_letter_map_from_frequency(ciphertext, english_freqs, verbose=False)
decoded_freq = decode(ciphertext, freq_map)

# Metropolis/bigram decode
bigram_stats = all_bigram_freqs(moby_new, character_list)
decoded_metro = metropolis(ciphertext, character_list, bigram_stats,
                           N=20000, T=1.0, verbose=False)

# build inferred mappings for each decode
map_freq  = get_mapping_from_decoded(ciphertext, decoded_freq)
map_metro = get_mapping_from_decoded(ciphertext, decoded_metro)

same_letters, diff_letters = compare_maps(map_freq, map_metro)
pos_agree = positional_overlap(decoded_freq, decoded_metro)

print("Extension 8 – frequency vs Metropolis")
print("-------------------------------------")
print(f"Message length: {len(ciphertext)}")
print(f"Same letter mappings : {same_letters}")
print(f"Different mappings   : {diff_letters}")
print(f"Positional agreement : {pos_agree*100:.2f}%")

print("\n--- First 300 chars of frequency-based decode ---")
print(decoded_freq[:300])

print("\n--- First 300 chars of Metropolis decode ---")
print(decoded_metro[:300])
print("\n--------------------------------------------")